<img src="../assets/tumor_twin.png" alt="Tumor Twin" width="500"/>

# Avascular PDE demo: HGG data + 4-field tumor model

This tutorial mirrors the diffusion and immune demos, but uses `AvascularTumorGrowth3D` with state `(n, m, h, s)`.

- `n`: proliferating tumor cells
- `m`: non-proliferating tumor cells
- `h`: healthy tissue
- `s`: nutrient

For measured ADC cellularity comparison, we use **total tumor**: `n + m`.


## Table of contents
- Optional: Google Colab
- Imports & paths
- Load patient & ADC-derived cellularity
- Build avascular model
- Forward solve
- Predicted vs measured TCC (`n+m`)
- Residuals at visits
- Calibration (LM)
- Summary


In [ ]:
# Optional: Google Colab setup
import sys
import importlib.util
from pathlib import Path

if "google.colab" in sys.modules:
    get_ipython().system("pip uninstall -y torchvision torchaudio thinc fastai")
    def is_package_installed(package_name):
        return importlib.util.find_spec(package_name) is not None
    if not is_package_installed("tumortwin"):
        get_ipython().system("pip install git+https://github.com/OncologyModelingGroup/TumorTwin")
    data_path = Path("../input_files/HGG_demo_001")
    if not data_path.exists():
        get_ipython().system("wget https://github.com/OncologyModelingGroup/TumorTwin/raw/refs/heads/main/input_files/HGG_demo_001.tar.gz")
        get_ipython().system("tar -xzvf HGG_demo_001.tar.gz")
        get_ipython().system("mkdir -p ../input_files")
        get_ipython().system("mv HGG_demo_001 ../input_files/HGG_demo_001")


### Imports & paths


In [ ]:
import gc
import matplotlib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from datetime import timedelta
from pathlib import Path

from pydantic import FilePath

from tumortwin.models import AvascularTumorGrowth3D
from tumortwin.models.pde_system import extract_trajectory_component
from tumortwin.optimizers import LMoptimizer, LMoptions
from tumortwin.pde_workflow import (
    select_timepoint_indices,
    fields_at_times_from_trajectory,
    spatiotemporal_residual_vector,
    squared_error_loss,
)
from tumortwin.postprocessing import (
    plot_cellularity_map,
    plot_imaging_summary,
    plot_measured_TCC,
    plot_patient_timeline,
    plot_predicted_TCC,
    plot_calibration_iter,
    plot_loss,
)
from tumortwin.preprocessing import ADC_to_cellularity, compute_carrying_capacity
from tumortwin.solvers import TorchDiffEqSolver, TorchDiffEqSolverOptions
from tumortwin.types import CropSettings, CropTarget
from tumortwin.types.hgg_data import HGGPatientData
from tumortwin.utils import daterange, days_since_first

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

%matplotlib inline
matplotlib.rc("font", size=10)
matplotlib.rc("figure", dpi=250)


### Load patient & ADC-derived cellularity


In [ ]:
data_path = Path("../input_files/HGG_demo_001")
PATIENT_INFO_PATH = FilePath(str(data_path / "HGG_demo_001.json"))
IMAGE_PATH = FilePath(str(data_path))
crop_settings = CropSettings(crop_to=CropTarget.ROI_ENHANCE, padding=10, visit_index=-1)

patient_data = HGGPatientData.from_file(
    PATIENT_INFO_PATH, image_dir=IMAGE_PATH, crop_settings=crop_settings
)

plot_patient_timeline(patient_data)
plt.show()
plot_imaging_summary(patient_data)
plt.show()


In [ ]:
measured_cellularity_maps = [
    ADC_to_cellularity(visit.adc_image, visit.roi_enhance_image, visit.roi_nonenhance_image)
    for visit in patient_data.visits
]
carrying_capacity = compute_carrying_capacity(patient_data.brainmask_image)
print("visits:", len(measured_cellularity_maps), "carrying capacity:", carrying_capacity)


### Build avascular model


In [ ]:
initial_n = torch.from_numpy(measured_cellularity_maps[0].array).float().to(device)
initial_m = torch.zeros_like(initial_n)
initial_h = torch.clamp(1.0 - initial_n, min=0.0, max=1.0)
initial_s = torch.ones_like(initial_n)

# HGG-compatible stable baseline for avascular model (article-inspired, scaled for day-based torchdiffeq timeline)
article_params = {
    "B": 0.14,
    "L": -0.5,
    "Dn": 2.6e-3,
    "Ds": 77.8,
    "mu": 2.0,
    "q_s": 3.0e-5,
    "K": 1.2,
    "s_0": 3.134e-2,
    "s_x": 0.418,
    "s_K": 1.0,
    "g_0": 1.0,
}

model = AvascularTumorGrowth3D(
    B=torch.tensor(article_params["B"], device=device),
    L=torch.tensor(article_params["L"], device=device),
    Dn=torch.tensor(article_params["Dn"], device=device),
    Ds=torch.tensor(article_params["Ds"], device=device),
    mu=torch.tensor(article_params["mu"], device=device),
    q_s=torch.tensor(article_params["q_s"], device=device),
    K=torch.tensor(article_params["K"], device=device),
    s_0=torch.tensor(article_params["s_0"], device=device),
    s_x=torch.tensor(article_params["s_x"], device=device),
    s_K=torch.tensor(article_params["s_K"], device=device),
    g_0=torch.tensor(article_params["g_0"], device=device),
    patient_data=patient_data,
    initial_n=initial_n,
    initial_m=initial_m,
    initial_h=initial_h,
    initial_s=initial_s,
    poisson_iterations=12,
    require_grad=True,
    device=device,
)

u0 = model.get_initial_state()
print("Avascular baseline params:", article_params)
print("Initial state shape (C,D,H,W):", tuple(u0.shape))


In [ ]:
# Pre-solve validator: shapes, finite values, parameter sanity

def _scalar(x):
    return float(x.detach().cpu().item())

required_shape = tuple(initial_n.shape)
assert u0.shape[0] == 4, f"Expected 4 state channels [n,m,h,s], got {u0.shape[0]}"
assert tuple(u0.shape[1:]) == required_shape, (
    f"State spatial shape mismatch: u0 is {tuple(u0.shape[1:])}, expected {required_shape}"
)

for name, field in {"n": u0[0], "m": u0[1], "h": u0[2], "s": u0[3]}.items():
    assert field.device == device, f"{name} is on {field.device}, expected {device}"
    assert torch.isfinite(field).all(), f"{name} contains NaN/Inf"

for pname in ["B", "L", "Dn", "Ds", "mu", "q_s", "K", "s_0", "s_x", "s_K", "g_0"]:
    p = getattr(model, pname)
    assert torch.isfinite(p).all(), f"Parameter {pname} contains NaN/Inf"

assert _scalar(model.L) < 0.0, f"Strict article mode requires L < 0, got {_scalar(model.L)}"
assert _scalar(model.Dn) > 0.0, f"Dn must be > 0, got {_scalar(model.Dn)}"
assert _scalar(model.Ds) > 0.0, f"Ds must be > 0, got {_scalar(model.Ds)}"
assert _scalar(model.s_0) > 0.0, f"s_0 must be > 0, got {_scalar(model.s_0)}"
assert _scalar(model.s_x) > 0.0, f"s_x must be > 0, got {_scalar(model.s_x)}"
assert _scalar(model.s_K) > 0.0, f"s_K must be > 0, got {_scalar(model.s_K)}"
assert getattr(model, "time_scale_days", 30.0) > 0.0, "time_scale_days must be > 0"

# Practical stability hints (article-inspired ranges adapted to day-based solve)
if not (0.01 <= _scalar(model.B) <= 1.0):
    print(f"[WARN] B={_scalar(model.B):.3g} is outside typical stable range [0.01, 1.0].")
if not (1e-7 <= _scalar(model.q_s) <= 1e-3):
    print(f"[WARN] q_s={_scalar(model.q_s):.3g} is outside typical stable range [1e-7, 1e-3].")

print("Validator OK")
print("u0 shape:", tuple(u0.shape))
print("u0 device/dtype:", u0.device, u0.dtype)
print(
    "params:",
    {
        "B": _scalar(model.B),
        "L": _scalar(model.L),
        "Dn": _scalar(model.Dn),
        "Ds": _scalar(model.Ds),
        "mu": _scalar(model.mu),
        "q_s": _scalar(model.q_s),
        "K": _scalar(model.K),
        "s_0": _scalar(model.s_0),
        "s_x": _scalar(model.s_x),
        "s_K": _scalar(model.s_K),
        "g_0": _scalar(model.g_0),
        "time_scale_days": float(getattr(model, "time_scale_days", 30.0)),
    },
)
print(
    "u0 max:",
    {
        "n": float(u0[0].max().detach().cpu()),
        "m": float(u0[1].max().detach().cpu()),
        "h": float(u0[2].max().detach().cpu()),
        "s": float(u0[3].max().detach().cpu()),
    },
)

In [ ]:
solver = TorchDiffEqSolver(
    model,
    TorchDiffEqSolverOptions(
        step_size=timedelta(days=0.25),
        use_adjoint=False,
        device=device,
        method="rk4",
    ),
)


### Forward solve


In [ ]:
# Mini end-to-end sanity evaluation (30-day window)
from datetime import timedelta

mini_t0 = patient_data.visits[0].time
mini_timepoints = [mini_t0 + timedelta(days=d) for d in range(0, 31)]  # 0..30 days, daily output

_old_step = solver.solver_options.step_size
_old_method = solver.solver_options.method
_old_adj = solver.solver_options.use_adjoint
_old_pi = model.poisson_iterations

solver.solver_options.method = "rk4"
solver.solver_options.use_adjoint = False
solver.solver_options.step_size = timedelta(days=0.05)
model.poisson_iterations = 12

with torch.no_grad():
    _, traj_mini = solver.solve(timepoints=mini_timepoints, u_initial=u0)

solver.solver_options.step_size = _old_step
solver.solver_options.method = _old_method
solver.solver_options.use_adjoint = _old_adj
model.poisson_iterations = _old_pi

print("traj shape:", tuple(traj_mini.shape))
print("finite:", bool(torch.isfinite(traj_mini).all()))
print("any inf:", bool(torch.isinf(traj_mini).any()))
print("any nan:", bool(torch.isnan(traj_mini).any()))

n_mini = extract_trajectory_component(traj_mini, 0)
m_mini = extract_trajectory_component(traj_mini, 1)
h_mini = extract_trajectory_component(traj_mini, 2)
s_mini = extract_trajectory_component(traj_mini, 3)
tumor_mini = torch.clamp(n_mini + m_mini, min=0.0)

print(
    "max n/m/h/s:",
    float(n_mini.max()),
    float(m_mini.max()),
    float(h_mini.max()),
    float(s_mini.max()),
)
print("max n+m:", float(tumor_mini.max()))
print("max n+m+h:", float((n_mini + m_mini + h_mini).max()))

if float((n_mini + m_mini + h_mini).max()) > 1.05:
    print("[WARN] local occupancy exceeds 1.05; reduce B/mu or step_size.")
else:
    print("[PASS] occupancy check looks good.")

In [ ]:
# Use HALF of the patient timeline (full cycle on truncated interval)
t_start = patient_data.visits[0].time
t_stop_full = patient_data.visits[-1].time
half_end = t_start + (t_stop_full - t_start) / 2

timepoints = daterange(t_start, half_end, timedelta(days=0.5))

# Keep only measured visits within the half interval
visits_half = [v for v in patient_data.visits if v.time <= half_end]
if len(visits_half) < 2:
    raise RuntimeError("Half-interval contains <2 visits. Increase interval or adjust data.")
measured_cellularity_maps_half = measured_cellularity_maps[: len(visits_half)]
visit_times_half = [v.time for v in visits_half]
visit_days_half = [days_since_first(v.time, t_start) for v in visits_half]

print("Half interval:", t_start, "->", half_end)
print("Forward timepoints:", len(timepoints), "| measured visits in half:", len(visits_half))
print(
    "Diagnostics: comp_mask voxels =",
    float(model.comp_mask.sum()),
    "| u0 channel n min/max =",
    float(u0[0].min()),
    float(u0[0].max()),
)

_, trajectory = solver.solve(timepoints=timepoints, u_initial=u0)
print("Trajectory shape:", tuple(trajectory.shape))

n_traj = extract_trajectory_component(trajectory, 0)
m_traj = extract_trajectory_component(trajectory, 1)
tumor_traj = torch.clamp(n_traj + m_traj, min=0.0)
if not torch.isfinite(trajectory).all():
    raise RuntimeError("Forward trajectory contains NaN/Inf; reduce step_size and/or lower B,Dn,mu")

tumor_maps = [tumor_traj[i] for i in range(tumor_traj.shape[0])]
print("Finite check passed. tumor min/max:", float(torch.nan_to_num(tumor_traj).min()), float(torch.nan_to_num(tumor_traj).max()))
print(
    "tumor n+m at t0 / t_end max:",
    float(tumor_traj[0].max()),
    float(tumor_traj[-1].max()),
)


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 2.5))
plot_predicted_TCC(tumor_maps, timepoints, ax=ax, carrying_capacity=carrying_capacity)
plot_measured_TCC([m.array for m in measured_cellularity_maps_half], visit_times_half, ax=ax)
ax.legend(["predicted (n+m)", "measured (ADC)"])
ax.set_title("Avascular model (half interval): predicted vs measured TCC")
plt.tight_layout()
plt.show()


In [ ]:
# Spatial comparison at visits inside the half interval
visit_days = visit_days_half
t_tensor = torch.tensor([days_since_first(t, timepoints[0]) for t in timepoints], dtype=torch.float32, device=device)
idx_vis = select_timepoint_indices(t_tensor, visit_days, atol=0.51)

fig, axes = plt.subplots(2, len(visit_days), figsize=(3 * len(visit_days), 5), squeeze=False)
for i, vd in enumerate(visit_days):
    plot_cellularity_map(tumor_maps[idx_vis[i]].cpu(), patient_data, time=vd, ax=axes[0, i])
    plot_cellularity_map(torch.tensor(measured_cellularity_maps_half[i].array).float(), patient_data, time=vd, ax=axes[1, i])
axes[0, 0].set_ylabel("Predicted n+m")
axes[1, 0].set_ylabel("Measured")
plt.tight_layout()
plt.show()


### Residuals at visit times


In [ ]:
n_visits_cal = min(4, len(visits_half))
visit_days_cal = visit_days_half[:n_visits_cal]
idx_cal = select_timepoint_indices(t_tensor, visit_days_cal, atol=0.51)

pred_maps = [tumor_traj[i] for i in idx_cal]
meas_maps = [
    torch.tensor(measured_cellularity_maps_half[j].array, dtype=torch.float32, device=device)
    for j in range(n_visits_cal)
]
res = spatiotemporal_residual_vector(pred_maps, meas_maps)
loss = squared_error_loss(res)
print("Residual length:", res.numel(), "SSE:", float(loss.detach().cpu()))


### Calibration (LM)


In [ ]:
n_visits_calibration = min(4, len(visits_half))
calibration_timepoints = [v.time for v in visits_half[:n_visits_calibration]]
target_solution = torch.stack(
    tuple(torch.from_numpy(measured_cellularity_maps_half[i].array).float() for i in range(n_visits_calibration))
).to(device)
print("Calibration target shape:", tuple(target_solution.shape))
print("Calibration visits (half interval):", n_visits_calibration)

_max_times_viz = 28
print("timepoints cap for calibration plots:", _max_times_viz)


In [ ]:
def update_avascular_predict(model_parameters, timepoints=None):
    if timepoints is None:
        timepoints = calibration_timepoints

    Bv = float(model_parameters[0].item() if hasattr(model_parameters[0], "item") else model_parameters[0])
    Dnv = float(model_parameters[1].item() if hasattr(model_parameters[1], "item") else model_parameters[1])
    muv = float(model_parameters[2].item() if hasattr(model_parameters[2], "item") else model_parameters[2])

    m = solver.model
    m.B = nn.Parameter(torch.tensor(Bv, device=device, dtype=torch.float32))
    m.Dn = nn.Parameter(torch.tensor(Dnv, device=device, dtype=torch.float32))
    m.mu = nn.Parameter(torch.tensor(muv, device=device, dtype=torch.float32))

    _, traj = solver.solve(timepoints=timepoints, u_initial=m.get_initial_state())
    n_t = extract_trajectory_component(traj, 0)
    m_t = extract_trajectory_component(traj, 1)
    return torch.clamp(n_t + m_t, min=0.0)


# Calibration around stable avascular baseline (B~0.14, Dn~2.6e-3, mu~2)
initial_cal_params = torch.tensor((0.14, 2.6e-3, 2.0), dtype=torch.float32)
cal_optim = LMoptimizer(
    model=update_avascular_predict,
    initial_guess=initial_cal_params,
    bounds=torch.tensor(((0.02, 0.8), (1.0e-4, 1.0e-2), (0.1, 4.0))),
    y_data=target_solution,
    options=LMoptions(),
)


In [ ]:
n_cal_iter = 5
for it in range(n_cal_iter):
    print(f"Calibration iteration {it + 1}/{n_cal_iter}")
    cal_optim.step()

best_p = cal_optim.best_x.clone()
print("Best parameters (B, Dn, mu):", best_p)


In [ ]:
_viz_stride = max(1, len(timepoints) // _max_times_viz)
timepoints_viz = timepoints[::_viz_stride]

n_snap = min(2, len(cal_optim.parameters))
snap_step = max(1, len(cal_optim.parameters) // n_snap)
calibration_sols = []
for p in cal_optim.parameters[::snap_step]:
    calibration_sols.append(update_avascular_predict(p, timepoints=timepoints_viz))
    gc.collect()

fig, ax = plt.subplots(1, 1, figsize=(6, 3))
plot_calibration_iter(
    sols=calibration_sols,
    carrying_capacity=carrying_capacity,
    timepoints=timepoints_viz,
    measured_cellularity_maps=measured_cellularity_maps_half,
    patient_data=patient_data,
    t_calibration_end=calibration_timepoints[-1],
    ax=ax,
)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(5, 2.5))
plot_loss(torch.tensor(cal_optim.error, dtype=torch.float64), ax=ax)
plt.tight_layout()
plt.show()

del calibration_sols
gc.collect()


In [ ]:
_ = update_avascular_predict(best_p, timepoints=timepoints_viz)
_, traj_cal = solver.solve(timepoints=timepoints_viz, u_initial=solver.model.get_initial_state())

n_cal = extract_trajectory_component(traj_cal, 0)
m_cal = extract_trajectory_component(traj_cal, 1)
tumor_maps_cal = [torch.clamp(n_cal[i] + m_cal[i], min=0.0) for i in range(n_cal.shape[0])]

fig, ax = plt.subplots(1, 1, figsize=(5, 2.5))
plot_predicted_TCC(tumor_maps_cal, timepoints_viz, ax=ax, carrying_capacity=carrying_capacity, color="tab:blue")
plot_measured_TCC([m.array for m in measured_cellularity_maps_half], visit_times_half, ax=ax)
ax.legend(["predicted calibrated (n+m)", "measured (ADC)"])
ax.set_title("Avascular model (half interval) TCC after LM")
plt.tight_layout()
plt.show()


## Summary

- This demo follows the same structure as diffusion and immune tutorials.
- `AvascularTumorGrowth3D` solves a 4-field PDE system `(n,m,h,s)`.
- For measured cellularity comparison, we used total tumor `n+m`.
- Calibration uses LM on voxel maps at visit times and memory-safe plotting via a strided timeline.
